In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ai4privacy/pii-masking-400k")

In [ ]:
train_ds = ds['train'].filter(lambda x: x['language'] == 'es')['source_text']
dev_ds = ds['validation'].filter(lambda x: x['language'] == 'es')['source_text']

In [ ]:
ds

In [ ]:
for item in train_ds:
    print(item)
    break

In [ ]:
API_URL = "http://localhost:8000"  # Url for debugger. change it to your own

## Inference


In [ ]:
# Function to make single inference using the API
def get_predictions(sample: str) -> dict:
    response = requests.post(url=f"{API_URL}/anonymizer/predict", json={"text": sample}, params={"use_cache": False})
    response.raise_for_status()
    return response.json()

In [ ]:
predictions = get_predictions(train_ds[0])

In [ ]:
train_ds[0]

In [ ]:
from pathlib import Path
import json

OUT_PATH = Path('../../../resources/data/experiments/ner_langextract_alignment/hf_pii_masking_400k_es_train.jsonl')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def _to_py_scalar(value):
    return value.item() if hasattr(value, 'item') else value

train_rows = ds['train'].filter(lambda x: x['language'] == 'es')

written = 0
with OUT_PATH.open('w', encoding='utf-8') as f:
    for idx, row in enumerate(train_rows):
        text = str(row.get('source_text') or '').replace('\x00', '').strip()
        if not text:
            continue

        uid = _to_py_scalar(row.get('uid'))
        record = {
            'sample_id': str(uid) if uid is not None else f'hf-train-{idx}',
            'document_id': 'ai4privacy/pii-masking-400k',
            'paragraph_id': str(idx),
            'source_path': 'hf://ai4privacy/pii-masking-400k/train',
            'text': text,
        }

        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        written += 1

print(f'Wrote {written} records to: {OUT_PATH.resolve()}')
